# 05 · Encoder y decoder
*Las dos pilas, la máscara causal, la cross-attention y un transformer que invierte secuencias*

Notebook del episodio 5 de **Attention is all you need** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/attention-is-all-you-need/arquitectura)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** diagrama de la arquitectura (N = 6) → nn.Transformer con los números del paper (44,1 M sin embeddings) → tres usos de la atención → log p(y|x) = Σ log p(y_i|y_<i, x), BOS + máscara → generate_square_subsequent_mask = triu → cross-attention: A de T_dec × T_enc → Bahdanau vs cross-attention → entrenar: 97 % invirtiendo 16 dígitos (GRU ep. 1: 69 %) → generar greedy token por token

**Cómo correrlo:** con el entorno de `~/Movies/serie-attention/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "attention"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Attention is all you need. Episodio cinco: encoder y decoder. Ya tenemos las piezas: la atención, las heads y el positional encoding. Hoy las armamos como en el paper: dos pilas, una máscara y una atención que cruza de una pila a la otra. Y al final, un transformer chico que aprende a invertir secuencias, donde la GRU, gated recurrent unit, del episodio uno se trababa.*

## Parte 1 · Las dos pilas

El transformer del paper es un traductor: un encoder que lee la frase de entrada y un decoder que escribe la traducción. Como en seq2seq, pero sin recurrencia.

### La arquitectura del paper

El diagrama del paper, en cajas. El encoder: los tokens de entrada, con su embedding más el positional encoding, pasan por una self-attention, donde todos se miran con todos, y por una red feed-forward que procesa cada token por separado. Ese bloque se repite seis veces. Lo que sale es la memoria: un vector por token de entrada, no uno solo como en seq2seq. El decoder recibe la salida escrita hasta ahora, corrida un lugar. Tiene tres partes: una self-attention con máscara, una cross-attention que mira la memoria del encoder, y su feed-forward. También seis veces. Y al final, una capa lineal y un softmax dan la probabilidad del próximo token. La feed-forward, las residuales y la normalización las vemos en el episodio seis.

entrada + PE → self-attention → feed-forward → ×6: encoder → memoria → salida corrida + PE → masked self-attn → cross-attention → feed-forward → lineal + softmax

> **Vaswani et al., 2017 — Attention Is All You Need, sección 3.1** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **N = 6 · d_model = 512** · «The encoder is composed of a stack of N = 6 identical layers.» El decoder también, y «inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack».

La sección tres punto uno del paper. El encoder es una pila de N igual a seis capas idénticas, cada una con dos subcapas: multi-head self-attention y una red feed-forward. Todas producen vectores de d model igual a quinientos doce. El decoder es otra pila de seis capas, que agrega una tercera subcapa: multi-head attention sobre la salida del encoder.

### El modelo base, en nn.Transformer

PyTorch trae esta arquitectura armada. La configuramos con los números del paper: d model quinientos doce, ocho heads, seis capas de cada lado, y feed-forward de dos mil cuarenta y ocho. Tomamos la primera capa de cada pila. Recorremos las dos. Y listamos sus atenciones y sus capas lineales. Seis y seis capas. El encoder tiene self attn y dos lineales, que son la feed-forward. El decoder tiene una atención más: multihead attn, que es la cross-attention. Cuarenta y cuatro coma un millones de parámetros, sin contar los embeddings.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_capas.py
__file__ = 'snippets/ep05_capas.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch.nn as nn

In [3]:
tr = nn.Transformer(d_model=512, nhead=8, num_encoder_layers=6,
                    num_decoder_layers=6, dim_feedforward=2048)
enc, dec = tr.encoder.layers[0], tr.decoder.layers[0]
print("encoder:", len(tr.encoder.layers), "capas · decoder:",
      len(tr.decoder.layers), "capas")
for nombre, capa in [("capa del encoder", enc), ("capa del decoder", dec)]:
    partes = [n for n, _ in capa.named_children()
              if "attn" in n or n.startswith("linear")]
    print(f"{nombre}:", ", ".join(partes))
total = sum(p.numel() for p in tr.parameters())
print(f"parámetros (sin embeddings): {total / 1e6:.1f} M")

encoder: 6 capas · decoder: 6 capas
capa del encoder: self_attn, linear1, linear2
capa del decoder: self_attn, multihead_attn, linear1, linear2
parámetros (sin embeddings): 44.1 M


### Tres usos de la misma atención

- Encoder: self-attention, todos con todos
- Decoder: self-attention con máscara, solo hacia atrás
- Encoder-decoder: queries del decoder, keys y values del encoder
- Es la misma función del episodio 2; cambia de dónde salen Q, K y V

La sección tres punto dos punto tres del paper lista tres usos de multi-head attention. En el encoder, self-attention: query, key y value salen de la misma capa anterior, y cada token mira a todos. En el decoder, self-attention con máscara: cada posición mira solo a las anteriores y a sí misma. Y la encoder-decoder attention, que hoy se suele llamar cross-attention: las queries vienen del decoder, y las keys y values, de la memoria del encoder. Las tres son la misma función que escribimos en el episodio dos. Lo único que cambia es de dónde salen Q, K y V, y si hay máscara.

## Parte 2 · La máscara causal

El decoder escribe de a un token, pero se entrena con la frase entera de una vez. La máscara es lo que hace que las dos cosas sean compatibles.

### Entrenar en paralelo sin hacer trampa

$$
\log p(y \mid x) = \textstyle\sum_{i=1}^{T} \log p\big(y_i \mid y_{<i},\, x\big)
$$
<small>la probabilidad de la frase: un término por token</small>

$$
y^{\text{in}} = (\langle\text{BOS}\rangle,\, y_1,\, \dots,\, y_{T-1})
$$
<small>la entrada del decoder: la salida corrida un lugar</small>

$$
M_{ij} = -\infty \;\text{ si } j > i
$$
<small>la posición $i$ no ve $y_i$ ni lo que sigue</small>

El objetivo es predecir la salida token por token. La probabilidad de la frase entera es el producto de las de cada token dado los anteriores y la entrada x; con logaritmos, una suma. y menor que i son los tokens de antes. En el dibujo, cuatro términos que se suman. Para entrenarlos todos a la vez, al decoder se le da la salida correcta corrida un lugar, con un token especial de inicio, BOS, adelante. En el dibujo: para predecir cero, dos, siete, ocho, entra BOS, cero, dos, siete. La posición i tiene que predecir y sub i. Pero la self-attention vería y sub i en la posición siguiente: estaría copiando la respuesta. La máscara pone menos infinito en el futuro. En el dibujo, la fila de BOS solo se ve a sí misma.

### La máscara de PyTorch

nn punto Transformer trae la máscara hecha: generate square subsequent mask. La comparamos con la nuestra del episodio dos, la booleana de triu. Con torch punto equal. La de PyTorch es de números, no booleana: ceros donde se puede mirar y menos infinito donde no. Se suma a los scores antes del softmax, que es la misma cuenta que nuestro masked fill. Y marcan exactamente las mismas posiciones: True.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_causal.py
__file__ = 'snippets/ep05_causal.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn

In [5]:
M = nn.Transformer.generate_square_subsequent_mask(4)
print(M)                                  # float: 0 o -inf
nuestra = torch.triu(torch.ones(4, 4, dtype=torch.bool), 1)
print("misma máscara:", torch.equal(M == float("-inf"), nuestra))

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
misma máscara: True


## Parte 3 · Cross-attention

La pieza que conecta las dos pilas. Es la heredera directa de la atención de Bahdanau del episodio uno.

### Q del decoder, K y V del encoder

$$
Q = H_{\text{dec}} W^Q,\quad K = M\, W^K,\quad V = M\, W^V
$$
<small>$M$: la memoria, una fila por token de entrada</small>

$$
A = \mathrm{softmax}\Big(\frac{Q K^\top}{\sqrt{d_k}}\Big) \in \mathbb{R}^{T_{\text{dec}} \times T_{\text{enc}}}
$$
<small>sin máscara: la entrada entera ya se conoce</small>

$$
\text{out} = A\, V:\; (3 \times 5)(5 \times d) = 3 \times d
$$
<small>cada token del decoder, un promedio de la entrada</small>

H dec es la matriz de estados del decoder, una fila por token ya escrito; M, la memoria del encoder, una fila por token de entrada. Las queries salen de H dec; las keys y los values, de M. En el dibujo, tres tokens escritos contra cinco de entrada: la tabla de scores ya no es cuadrada, es de tres por cinco. Softmax por filas, y sin máscara: la frase de entrada se conoce entera desde el principio. En el dibujo, la fila cero real del próximo snippet. Y A por V: cada token del decoder recibe un promedio de los values de la entrada. Es la atención de Bahdanau con dos cambios: producto punto en vez de una red chica, y varias heads.

### Cross-attention con nuestra función

Una memoria de cinco tokens de entrada, de ocho números cada uno. Y tres estados del decoder. Las queries, del decoder. Las keys y los values, de la memoria. Y la misma attention del episodio dos, sin máscara. Q de tres por ocho, K y V de cinco por ocho. A es de tres por cinco: tres filas, una por token escrito, que reparten su peso entre los cinco de la entrada. La salida tiene una fila por token del decoder. La fila cero: cero coma quince, cero coma once, cero coma veintinueve, cero coma veintiuno y cero coma veinticinco.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_cross.py
__file__ = 'snippets/ep05_cross.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
torch.manual_seed(0)
d = 8


def attention(Q, K, V, mask=None):
    S = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        S = S.masked_fill(mask, float("-inf"))
    A = torch.softmax(S, dim=-1)
    return A @ V, A


W_q, W_k, W_v = (torch.randn(d, d) / d ** 0.5 for _ in range(3))

In [7]:
memoria = torch.randn(5, d)        # salida del encoder: 5 tokens de entrada
dec = torch.randn(3, d)            # estado del decoder: 3 tokens escritos

Q = dec @ W_q                      # queries: del decoder
K, V = memoria @ W_k, memoria @ W_v   # keys y values: del encoder
out, A = attention(Q, K, V)        # sin máscara: la entrada es toda conocida
print("Q", tuple(Q.shape), " K", tuple(K.shape), " V", tuple(V.shape))
print("A", tuple(A.shape), " out", tuple(out.shape))
print("fila 0 de A:", A[0].numpy().round(2))

Q (3, 8)  K (5, 8)  V (5, 8)
A (3, 5)  out (3, 8)
fila 0 de A: [0.15 0.11 0.29 0.21 0.25]


### De Bahdanau a la cross-attention

La comparación con el episodio uno. En Bahdanau, la query es el estado de la RNN, la red recurrente, del decoder, las keys y los values son los mismos estados del encoder, y el puntaje lo da una red chica con tanh. En la cross-attention, la query es la fila del decoder, keys y values son proyecciones distintas de la memoria, el puntaje es un producto punto, y hay ocho heads en cada una de las seis capas. Misma idea, más barata y más expresiva.

## Parte 4 · Un transformer que invierte

Volvemos a la tarea del episodio uno: invertir dieciséis dígitos. Ahí, la GRU sin atención se quedaba en sesenta y nueve por ciento.

### Entrenar el encoder-decoder

Un transformer chico: d model sesenta y cuatro, cuatro heads, dos capas de encoder y dos de decoder, con el positional encoding sinusoidal del episodio cuatro. Por dentro usa nn punto Transformer, que como el paper normaliza después de cada subcapa. Creamos el modelo. Cada paso, ciento veintiocho secuencias al azar: x, la salida y, y la entrada del decoder, BOS más y corrida. Un forward con la máscara causal adentro. La pérdida es la entropía cruzada contra y. Y medimos aciertos en mil secuencias nuevas. La pérdida baja de dos coma cuarenta y siete a cero coma cero noventa y cinco en ochocientos pasos. Aciertos por dígito: noventa y siete por ciento.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_entrenar.py
__file__ = 'snippets/ep05_entrenar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import os
import torch
import torch.nn as nn
torch.manual_seed(0)
torch.set_num_threads(4)
V, BOS, L, d = 10, 10, 16, 64          # dígitos 0–9, BOS = 10, largo 16
GUARDAR = os.path.join(os.path.dirname(__file__), "..", "datos", "ep05_inversor.pt")


def pe(n, d):
    p = torch.arange(n).unsqueeze(1); om = 1 / 10000 ** (torch.arange(0, d, 2) / d)
    out = torch.zeros(n, d); out[:, 0::2], out[:, 1::2] = torch.sin(p * om), torch.cos(p * om)
    return out


class Inversor(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V + 1, d)
        self.register_buffer("pe", pe(64, d))
        self.tr = nn.Transformer(d, nhead=4, num_encoder_layers=2, num_decoder_layers=2,
                                 dim_feedforward=128, dropout=0.0, batch_first=True)
        self.out = nn.Linear(d, V)

    def forward(self, x, y_in):
        T = y_in.shape[1]
        causal = nn.Transformer.generate_square_subsequent_mask(T)
        h = self.tr(self.emb(x) * math.sqrt(d) + self.pe[:x.shape[1]],
                    self.emb(y_in) * math.sqrt(d) + self.pe[:T], tgt_mask=causal)
        return self.out(h)


def lote(n):
    x = torch.randint(0, V, (n, L)); y = x.flip(1)
    return x, y, torch.cat([torch.full((n, 1), BOS), y[:, :-1]], 1)

In [9]:
modelo = Inversor()
opt = torch.optim.Adam(modelo.parameters(), lr=2e-3)
for paso in range(1, 801):
    x, y, y_in = lote(128)                 # y_in = BOS + y corrida 1
    logits = modelo(x, y_in)               # [128, 16, 10]
    loss = nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if paso in (1, 200, 400, 600, 800):
        print(f"paso {paso:3}: pérdida {loss.item():.3f}")

x, y, y_in = lote(1000)
with torch.no_grad():
    acierto = (modelo(x, y_in).argmax(-1) == y).float().mean()
print(f"L = {L}: aciertos por dígito = {acierto:.1%}")

paso   1: pérdida 2.474


paso 200: pérdida 1.587


paso 400: pérdida 0.884


paso 600: pérdida 0.502


paso 800: pérdida 0.095
L = 16: aciertos por dígito = 97.0%


In [10]:
# Verificación extra (corrió junto con el video)
torch.save(modelo.state_dict(), GUARDAR)

### Invertir 16 dígitos (salidas reales)

Lado a lado. La GRU del episodio uno, con seiscientos pasos, acertaba el sesenta y nueve por ciento. El transformer, con ochocientos, el noventa y siete. No es una comparación de laboratorio: distinto modelo, distinto learning rate, distinta cantidad de pasos. Pero la diferencia es de forma: el decoder ya no depende de un vector fijo; en cada paso puede ir a buscar el dígito que necesita a la memoria. El azar, diez por ciento.

- **GRU, ep. 1 (600 pasos)**: 69 %
- **transformer (800 pasos)**: 97 %
- **al azar**: 10 %

### Generar token por token

Al entrenar, todo va en paralelo. Al generar, no queda otra que ir de a uno. El encoder procesa la entrada una sola vez y deja la memoria. El decoder arranca solo con BOS. En cada paso corre sobre todo lo escrito hasta ahora, con la máscara causal y la memoria. Del último vector sale la distribución del próximo dígito; tomamos el más probable, que se llama greedy, y lo pegamos al final. Paso uno: cero. Paso dos: cero, dos. Paso tres: cero, dos, siete. Y en el paso dieciséis, la secuencia completa, igual a la entrada al revés, dígito por dígito.

**Cada línea, su término:**

- `memoria = modelo.tr.encoder(emb(x))` → $M = \mathrm{Enc}(x)$ — el encoder corre una sola vez
- `y = torch.tensor([[BOS]])` → $y_0 = \langle\text{BOS}\rangle$ — se arranca con el token de inicio
- `h = modelo.tr.decoder(emb(y), memoria` → $h = \mathrm{Dec}(y_{\le t},\, M)$ — el decoder relee todo lo escrito
- `siguiente = modelo.out(h[:, -1]).argmax(-1)` → $\hat y_{t+1} = \arg\max\, \mathrm{softmax}(W h_t)$ — greedy: el más probable, y se agrega

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep05_generar.py
__file__ = 'snippets/ep05_generar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import os
import torch
import torch.nn as nn
V, BOS, L, d = 10, 10, 16, 64
GUARDADO = os.path.join(os.path.dirname(__file__), "..", "datos", "ep05_inversor.pt")


def pe(n, d):
    p = torch.arange(n).unsqueeze(1); om = 1 / 10000 ** (torch.arange(0, d, 2) / d)
    out = torch.zeros(n, d); out[:, 0::2], out[:, 1::2] = torch.sin(p * om), torch.cos(p * om)
    return out


class Inversor(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V + 1, d)
        self.register_buffer("pe", pe(64, d))
        self.tr = nn.Transformer(d, nhead=4, num_encoder_layers=2, num_decoder_layers=2,
                                 dim_feedforward=128, dropout=0.0, batch_first=True)
        self.out = nn.Linear(d, V)


modelo = Inversor()
modelo.load_state_dict(torch.load(GUARDADO))   # lo entrena ep05_entrenar.py
modelo.eval()
torch.manual_seed(7)
x = torch.randint(0, V, (1, L))
emb = lambda t: modelo.emb(t) * math.sqrt(d) + modelo.pe[:t.shape[1]]
mascara = nn.Transformer.generate_square_subsequent_mask
texto = lambda t: "".join(map(str, t.tolist()))

In [12]:
memoria = modelo.tr.encoder(emb(x))          # el encoder corre UNA vez
y = torch.tensor([[BOS]])                    # el decoder arranca con BOS
with torch.no_grad():
    for t in range(L):
        causal = mascara(y.shape[1])               # triangular
        h = modelo.tr.decoder(emb(y), memoria, tgt_mask=causal)
        siguiente = modelo.out(h[:, -1]).argmax(-1)   # solo el último
        y = torch.cat([y, siguiente.unsqueeze(0)], 1)
        if t in (0, 1, 2, L - 1):
            print(f"paso {t + 1:2}: {texto(y[0, 1:])}")
print("entrada :", texto(x[0]))
print("al revés:", texto(x[0].flip(0)))

paso  1: 0
paso  2: 02
paso  3: 027
paso 16: 0278181897736125
entrada : 5216377981818720
al revés: 0278181897736125


### La salida crece de a un dígito (salida real)

El mismo proceso, en cajas. Arranca con BOS. El primer forward del decoder produce el cero. El segundo, mirando BOS y cero, produce el dos. El tercero, el siete. Y así, dieciséis forwards. La secuencia final es la entrada invertida. Fijate que generar sigue siendo secuencial, como la RNN: lo que el transformer volvió paralelo es el entrenamiento, y eso fue lo que lo hizo escalar.

BOS → 0 → 02 → 027 → … → 0278181897736125

### Entrenar contra generar

- Entrenar: toda la salida entra de una, corrida; un solo forward
- La máscara hace que cada posición prediga sin ver su respuesta
- Generar: un forward del decoder por token nuevo
- El encoder corre una vez; el decoder relee lo escrito en cada paso

Vale la pena separar los dos regímenes. Al entrenar, la salida correcta entra entera, corrida un lugar, y un solo forward calcula la pérdida de las dieciséis posiciones. Eso es lo que aprovecha la placa de video. La máscara es lo que permite hacerlo sin trampa: cada posición predice su token sin verlo. Al generar no tenemos la salida correcta: cada token nuevo necesita un forward del decoder. El encoder corre una sola vez, pero nuestro decoder vuelve a procesar todo lo escrito en cada paso. Los modelos de hoy guardan las keys y values de los pasos anteriores para no recalcularlos; es el caché que Gemma achica con GQA, grouped-query attention.

> **Vaswani et al., 2017 — Attention Is All You Need, sección 3.1** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **posición i ← solo < i** · «This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i.»

El paper lo resume en una frase. La máscara, junto con que los embeddings de salida están corridos una posición, asegura que la predicción para la posición i solo puede depender de las salidas conocidas en posiciones menores que i. Son exactamente las dos cosas que hicimos: BOS adelante y la máscara triangular.

## Resumiendo

- Encoder: 6 capas de self-attention + feed-forward → memoria
- Decoder: masked self-attention, cross-attention y feed-forward, ×6
- Entrenar: salida corrida + máscara causal → todo en paralelo
- Cross-attention: $Q$ del decoder, $K, V$ de la memoria; $A$ es $T_{dec} \times T_{enc}$
- Invertir 16 dígitos: 97 % (la GRU del ep. 1, 69 %); generar es de a uno
- Próximo: feed-forward, residuales y LayerNorm